# imatrix + IQ quantization on real devices (preview)

Runs importance-matrix (imatrix) guided quantization and the sub-4-bit IQ rungs
(IQ4_NL / IQ3_S / IQ2_S) on a real phone, each scored by on-device perplexity. This
is the on-device-legal analog of AWQ/GPTQ calibration, running on llama.cpp.

Installs the feature branch (not yet on PyPI).

**Before you run:** a [tinyedge.ai](https://tinyedge.ai) account + API key, a device paired in the TinyEdge Runner app with "Available for benchmarks" on, and internet enabled (Kaggle: right sidebar).

In [ ]:
%pip -q install "git+https://github.com/TinyEdgeAI/tinyedge-agent.git@lienert/tin-39-imatrix-guided-iq-sub-4-bit-quantization-for-the-on-device#subdirectory=sdk" huggingface_hub

In [ ]:
import tinyedge

# Paste your API key from tinyedge.ai (New benchmark > Your API key):
client = tinyedge.TinyEdge(api_key="tinyedge_sk_REPLACE_ME")

DEVICES = client.devices(online=True)   # devices online right now
print("optimizing for:", DEVICES)

## Optimize a small model with imatrix + IQ

A high-precision (f16) SmolLM2-135M and a WikiText corpus. The corpus drives both the
importance matrix and the on-device perplexity scoring. The ladder now includes the IQ
rungs, each quantized with the imatrix; `report.summary()` shows size and perplexity per
variant so you can see the low-bit rungs hold up.

(Small model on purpose: optimize() uploads each variant to benchmark it, so the variants
must fit the upload cap.)

In [ ]:
from huggingface_hub import hf_hub_download
import io, zipfile, pathlib, requests

F16 = hf_hub_download("bartowski/SmolLM2-135M-Instruct-GGUF", "SmolLM2-135M-Instruct-f16.gguf")

pathlib.Path("corpus").mkdir(exist_ok=True)
wiki = zipfile.ZipFile(io.BytesIO(requests.get(
    "https://huggingface.co/datasets/ggml-org/ci/resolve/main/wikitext-2-raw-v1.zip", timeout=120).content))
pathlib.Path("corpus/wiki.txt").write_bytes(wiki.read("wikitext-2-raw/wiki.test.raw")[:200_000])

report = tinyedge.optimize(client, F16, devices=DEVICES, dataset="corpus")
print(report.summary())